# **Mineral Prospectivity Project**
## 02b National Geochemical Database (NGDB) exploratory data analysis

goals:\
-create plots and maps to investigate distributions, correlations and spatial trends

### Part 1. import packages and identify local directory
a. import packages needed for data loading and analysis

In [49]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import glob
import os

data_path = "/Users/adbyerly/prospectivity_model/data/processed/"

### Part 2. load processed data

In [50]:
# csv - load all geochemical files from the processed folder into a dictionary
files = glob.glob(data_path + "NGDB/*.csv")
ngdb_dfs_processed={}

for file in files:
    name = os.path.basename(file).replace(".csv", "")
    df = pd.read_csv(file, low_memory=False)
    ngdb_dfs_processed[name] = df

ngdb_dfs_processed.keys()
print(ngdb_dfs_processed['Rock_Data'].head())

   lab_id job_id         submitter    date_sub field_id state        country  \
0  AAV161   HM42  Gott, Garland B.  19660721.0    CD008    ID  United States   
1  AAV162   HM42  Gott, Garland B.  19660721.0   CD008A    ID  United States   
2  AAV163   HM42  Gott, Garland B.  19660721.0    CD009    ID  United States   
3  AAV164   HM42  Gott, Garland B.  19660721.0   CD009A    ID  United States   
4  AAV165   HM42  Gott, Garland B.  19660721.0    CD010    ID  United States   

  original_datum spheroid  latitude  ...  struct_src dep_envirn source_rk  \
0          NAD27      NaN  47.47778  ...         NaN        NaN       NaN   
1          NAD27      NaN  47.47778  ...         NaN        NaN       NaN   
2          NAD27      NaN  47.47667  ...         NaN        NaN       NaN   
3          NAD27      NaN  47.47667  ...         NaN        NaN       NaN   
4          NAD27      NaN  47.47556  ...         NaN        NaN       NaN   

  metamrphsm facies_grd prep mesh_size Unnamed: 31  \
0 

In [51]:
# examine processed dataframes

# for name, df in ngdb_dfs_processed.items():
#     print(name)
#     print(df.head())
#     print(df.columns)
#     print(df.shape)
#     print("-" * 25)

### Part 3. investigate data and document observations

a. start with the "Rock Data" dataframe which contains information about the samples

In [ ]:
# set variable for this table
rockdata = ngdb_dfs_processed['Rock_Data']

print(rockdata.head())
print(rockdata.columns)
print(rockdata.dtypes)
print(rockdata.shape)

In [ ]:
# general observations about sample quantity and locations

print ('There are ' + str(len(rockdata)) + ' samples.')

# print(rockdata['depth'].nunique())
# print(rockdata['depth'].unique())

# observation --------
print('All samples are effectively from the surface.')

print('Samples come from ' + str(rockdata['locat_desc'].nunique()) + ' locations.')

In [ ]:
# identify types of rocks



# -------- these don't contain apparently useful information
# print(rockdata['source_rk'].nunique())
# print(rockdata['source_rk'].unique())
# print(rockdata['source_rk'].value_counts(dropna=False))

# print(rockdata['sample_src'].nunique())
# print(rockdata['sample_src'].unique())

# print(rockdata['metamrphsm'].nunique())
# print(rockdata['metamrphsm'].unique())
# -------- these don't contain apparently useful information



# 'xnddryclass' feature contains information about the rock type
print(rockdata['primeclass'].unique())
print(rockdata['xndryclass'].unique())
print(rockdata['xndryclass'].value_counts(dropna=False))

rockdata['xndryclass'].value_counts().plot(kind='bar')

In [ ]:
# examine ages of igneous rocks

mask = rockdata['xndryclass'] == 'igneous'
igneous = rockdata[mask]

print(igneous['geol_age'].nunique())
print(igneous['geol_age'].unique().tolist())

In [ ]:
print(igneous['spec_name'].unique().tolist())
print(igneous.columns)

b. major element chemistry in igneous subset

In [ ]:
# filter chemical analysis tables to igenous rocks

igneous_id = igneous['lab_id']
igneous_chem = {}

for d, df in ngdb_dfs_processed.items():
    igneous_chem[d] = df[df['lab_id'].isin(igneous_id)].copy()  # copy df, filter to igneous rock primary key values 


print(igneous_chem.keys())

In [ ]:
# examine direct gold concentration measurements
# from metadata, these are the geochemical data tables that contain direct gold measurements:

# - NAA
# - ICPAES
# - ICPMS
# - ES

In [ ]:
# NAA

NAA = igneous_chem['NAA']
print(NAA.columns.tolist())
print(NAA.columns[NAA.columns.str.contains('au')])
au = NAA['auppb_na']

print(au.describe())
plt.hist(au, range = (0,20), bins='auto')
plt.xlim(0, 20)
plt.show()

In [ ]:
# identify columns for elements that may occur with gold
# Silver (Ag), Tellurium (Te), Copper (Cu), Mercury (Hg), Cobalt (Co), Antimony (Sb), Arsenic (As), Lead (Pb), Zinc (Zn)

print('Relevant column names for pathfinder elements:')
# Silver (Ag): 
print(NAA.columns[NAA.columns.str.contains('ag')]) # no relevant columns

#Tellurium (Te):
print(NAA.columns[NAA.columns.str.contains('te')]) # no relevant columns

# Copper (Cu):
print(NAA.columns[NAA.columns.str.contains('cu')])

# Mercury (Hg):
print(NAA.columns[NAA.columns.str.contains('hg')])

# Cobalt (Co): 
print(NAA.columns[NAA.columns.str.contains('co')])

# Antimony (Sb):
print(NAA.columns[NAA.columns.str.contains('sb')])

# Arsenic (As):
print(NAA.columns[NAA.columns.str.contains('as')])

# Lead (Pb):
print(NAA.columns[NAA.columns.str.contains('pb')]) # no relevant columns

# Zinc (Zn):
print(NAA.columns[NAA.columns.str.contains('zn')])


# create dataframe that is a subset including only the relevant columns
pathfinders = NAA[
    ['lab_id',
     'auppb_na',
     'cuppm_na',
     'hgppm_na',
     'coppm_na',
     'sbppm_na',
     'asppm_na',
     'znppm_na']
    ]

print(pathfinders.head())
print(pathfinders.shape)

In [ ]:
# join pathfinders dataframe with location data for samples

pathfinders_loc = pathfinders.merge(rockdata[['lab_id', 'geometry']], on = 'lab_id', how = 'inner')
print(pathfinders_loc.head())

print(pathfinders_loc.columns)

In [ ]:
pathfinders_loc.describe()
# drop copper and mercury, there are no data
pathfinders_loc.drop(columns=['cuppm_na', 'hgppm_na'], inplace=True)

In [ ]:
# convert to geodataframe and export pathfinders dataframe
from shapely import wkt

pathfinders_loc["geometry"] = pathfinders_loc["geometry"].apply(
    lambda x: wkt.loads(x) if isinstance(x, str) else x
)

pathfinders_loc = gpd.GeoDataFrame(
    pathfinders_loc,
    geometry=pathfinders_loc["geometry"]
)

pathfinders_loc = pathfinders_loc.set_crs("EPSG:32611")
# print(pathfinders_loc.head())

pathfinders_loc.to_file(data_path + "/NGDB/ngdb_naa_pathfinders.geojson", driver = "GeoJSON")